# ONCOVA 최종 제출 코드: 유전자 변이로 암 아형 분류

- 팀: 1조 ONCOVA (김지연·김도엽·여은지·김새봄·박종부)
- 최종 제출 파일: `submission_v12_stackS3.csv` → Public 리더보드 **0.4072** (11/13 10:2x 제출, 최종 선택)
- 이 노트북 하나를 위에서부터 실행하면 데이터 불러오기 → 표기 통일 → 피처 → 학습 → 예측 → 제출 파일 저장 → 추론용 모델 저장까지 끝남
- 외부 데이터 없음. 사전학습 모델 없음. test 데이터는 학습·통계 계산에 쓰지 않음 (표기 통일 기준값 `HOT_CNT`도 train에서만 계산함)
- 형제 라벨(test 변이를 train과 대조해 라벨을 바꾸는 규칙)은 **쓰지 않음**

**모델 요약**
1. 표기 통일: 같은 변이를 다른 기호로 적은 것(`X`→`*`, `Nfs`→`fs`)을 맞추고, del/ins는 빼고, 같은 변이를 여러 위치로 적은 것은 train에서 가장 흔한 위치 하나만 남김
2. 환자 중복 정리: 변이가 완전히 같은데 라벨이 묶음/세부(KIPAN↔KIRC, GBMLGG↔LGG)로 갈린 기록을 한 명으로 합치고 세부 라벨을 붙임
3. 1단계 모델 두 개
   - XGBoost: 유전자별 변이 있음/없음(4,384) + 변이 유전자 수 + 핫스팟(같은 유전자·위치가 5명 이상) + 아미노산 치환쌍(21×21) 비율·개수
   - 텍스트 마이닝: 환자의 변이 목록을 문장처럼 보고 `BRAF`, `BRAF_V600E`, `BRAF@600` 단어를 TF-IDF로 바꿔 로지스틱 회귀
4. 아형별 비중 학습(STACK): 학습 데이터 안 3-fold로 두 모델의 확률을 만들고, 이를 입력으로 받는 로지스틱 회귀(balanced)가 아형마다 어느 모델을 더 믿을지 배움
5. 2단계 판정: 겹치는 라벨은 확률을 합쳐 묶음으로 먼저 고르고, 묶음 안에서 세부 아형을 로지스틱 회귀로 정함 (50% 기준)
6. 시드 평균: 위 과정을 시드 42·7·123으로 세 번 학습해 test 확률(1단계·2단계)을 평균 낸 뒤 판정

**실행 환경 (제출자 로컬)**
- macOS 15.7 (Apple Silicon, arm64), Python 3.14.6
- pandas 3.0.5, numpy 2.5.3, scipy 1.18.1, scikit-learn 1.9.1, xgboost 3.4.1, joblib 1.6.0
- CPU만 사용, 전체 실행 약 10분 (교차검증까지 켜면 약 25분 추가)

In [1]:
# 실행 환경 확인 (OS·라이브러리 버전 출력)
import platform, sys
import numpy, pandas, scipy, sklearn, xgboost, joblib
print('OS     :', platform.platform())
print('Python :', sys.version.split()[0])
for m in (pandas, numpy, scipy, sklearn, xgboost, joblib):
    print(f'{m.__name__:7s}:', m.__version__)

OS     : macOS-15.7.7-arm64-arm-64bit-Mach-O
Python : 3.14.6
pandas : 3.0.5
numpy  : 2.5.3
scipy  : 1.18.1
sklearn: 1.9.1
xgboost: 3.4.1
joblib : 1.6.0


## 0. 경로와 설정

- 데이터 경로는 이 셀 한 곳에서만 바꾸면 됨 (`DATA_DIR`에 `train.csv`, `test.csv`, `sample_submission.csv`)
- `RUN_CV=True`로 바꾸면 5-fold 교차검증 점수도 계산함 (제출 파일에는 영향 없음, 시간만 더 걸림)

In [2]:
import os, re, time
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = 'data'        # 입력: train.csv, test.csv, sample_submission.csv
OUT_DIR = 'output'       # 출력: 제출 파일, 추론용 모델 파일
os.makedirs(OUT_DIR, exist_ok=True)

SEEDS = [42, 7, 123]     # 시드 평균에 쓰는 시드 세 개 (첫 번째가 교차검증 기준 시드)
HOT_N = 5                # 핫스팟: 같은 '유전자_위치'가 학습 데이터에서 5명 이상
LR_C = 10.0              # 텍스트 마이닝 로지스틱 회귀 규제 (클수록 약함)
STAGE2_C = 0.005         # 2단계 로지스틱 회귀 규제 (강하게)
RUN_CV = False           # True면 5-fold 교차검증 점수 계산

## 1. 데이터 불러오기

- 모든 칸을 글자로 읽고, 빈 칸은 `WT`(변이 없음)로 채움
- test에 train과 같은 유전자 열을 같은 순서로 맞춤 (없는 열은 WT)
- 라벨 26개는 `LabelEncoder`로 숫자로 바꿈 (train 라벨에만 맞춤)

In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', dtype=str, encoding='utf-8')
test = pd.read_csv(f'{DATA_DIR}/test.csv', dtype=str, encoding='utf-8')
submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv', encoding='utf-8')

genes = [c for c in train.columns if c not in ('ID', 'SUBCLASS')]   # 유전자 열 4,384개
G = np.array(genes)
V_raw = train[genes].fillna('WT').values                                # train 변이 표 (환자 × 유전자)
Vt_raw = test.reindex(columns=genes, fill_value='WT').fillna('WT').values   # test도 같은 열 순서
le = LabelEncoder(); y = le.fit_transform(train['SUBCLASS'])            # 라벨 → 0~25
CL = list(le.classes_); N = len(y)
print('train', V_raw.shape, '/ test', Vt_raw.shape, '/ 아형', len(CL))

train (6201, 4384) / test (2546, 4384) / 아형 26


## 2. 변이 표기 통일

- 같은 변이를 데이터마다 다르게 적은 것을 하나로 맞춤
  - `R213X` → `R213*` (끊김 표기), `S269Lfs` → `S269fs` (밀림 표기)
  - `del`·`ins`·`dup` 표기는 형식이 제각각이라 뺌
  - 한 칸에 숫자만 다른 같은 변이(예: 여러 전사체 위치로 반복 기록)가 있으면, **train에서 가장 흔한 위치** 하나만 남김
- 기준이 되는 위치 빈도 `HOT_CNT`는 **train에서만** 계산하고, 같은 규칙을 train·test에 똑같이 적용함

In [4]:
def shape_of(t):
    return re.sub(r'\d+', '', t)           # 숫자만 지운 모양: 'R213*' → 'R*'

HOT_CNT = Counter()                          # train에서 '유전자_위치'가 나온 환자 수
for row in V_raw:
    for j in np.flatnonzero(row != 'WT'):
        for h in {f'{G[j]}_{n[0]}' for n in (re.findall(r'\d+', m) for m in row[j].split()) if n}:
            HOT_CNT[h] += 1

def normalize(V):
    out = V.copy(); r, c = np.nonzero(V != 'WT')
    for i, j in zip(r, c):
        groups = defaultdict(list)                                   # 숫자만 다른 같은 변이끼리 묶음
        for t in V[i, j].split():
            if re.search(r'del|ins|dup', t): continue                # del/ins/dup 제외
            t = re.sub(r'^([A-Z]\d+)X$', r'\1*', t)                   # X → *
            t = re.sub(r'^([A-Z]\d+)[A-Z]fs$', r'\1fs', t)            # Lfs → fs
            groups[shape_of(t)].append(t)
        keep = []
        for ts in groups.values():
            if len(set(ts)) > 1:                                     # 같은 변이가 여러 위치로 적힘 → train에서 흔한 위치 하나
                keep.append(max(ts, key=lambda t: HOT_CNT.get(f'{G[j]}_{(re.findall(r"[0-9]+", t) or ["x"])[0]}', 0)))
            else:
                keep += ts
        out[i, j] = ' '.join(keep) if keep else 'WT'                 # 다 빠지면 변이 없음
    return out

V, Vt = normalize(V_raw), normalize(Vt_raw)
print('통일 전후 변이 칸 수 (train):', int((V_raw != 'WT').sum()), '→', int((V != 'WT').sum()))

통일 전후 변이 칸 수 (train): 218893 → 218890


## 3. 피처 만들기

- `BIN`: 유전자별 변이 있음 1 / 없음 0 (4,384개)
- `NMUT`: 변이가 있는 유전자 수 (모델이 가장 많이 참고한 피처)
- `HOT`: '유전자_위치' 단어 목록 → 학습 fold 안에서 5명 이상 나온 것만 핫스팟으로 씀 (모델 안에서 고름)
- `SUB`: 아미노산 치환쌍 21×21(+밀림·기타) 개수를 비율과 log 개수로 (암을 만든 원인마다 남기는 흔적이 다름: 자외선, 흡연 등)
- `WORDS`: 텍스트 마이닝용 단어 (`BRAF`, `BRAF_V600E`, `BRAF@600`)
- 라벨을 쓰지 않는 계산이라 train·test에 같은 함수를 씀

In [5]:
AA = list('ACDEFGHIKLMNPQRSTVWY') + ['*']; AI = {a: i for i, a in enumerate(AA)}   # 아미노산 20개 + 끊김
MS = re.compile(r'^([A-Z])(\d+)([A-Z*])$')                                          # 한 글자 치환: V600E, R175*

def featurize(V):
    n = len(V); Cm = np.zeros((n, 443), np.float32); hot, words = [], []
    for i, row in enumerate(V):
        h, w = [], []
        for j in np.flatnonzero(row != 'WT'):
            g = G[j]; w.append(g)                                   # 단어 1: 유전자 이름
            for t in dict.fromkeys(row[j].split()):                 # 같은 칸의 중복 표기는 한 번만
                m = MS.match(t)
                if m and m.group(1) in AI and m.group(3) in AI:
                    Cm[i, AI[m.group(1)] * 21 + AI[m.group(3)]] += 1    # 치환쌍 (예: V→E)
                elif 'fs' in t: Cm[i, 441] += 1                         # 밀림
                else: Cm[i, 442] += 1                                   # 그 밖의 형식
                w.append(f'{g}_{t}'); nums = re.findall(r'\d+', t)      # 단어 2: 유전자_변이
                if nums:
                    h.append(f'{g}_{nums[0]}'); w.append(f'{g}@{nums[0]}')   # 핫스팟 후보, 단어 3: 유전자@위치
        hot.append(list(dict.fromkeys(h))); words.append(w)
    BIN = sp.csr_matrix((V != 'WT').astype(np.float32))
    tot = Cm.sum(1, keepdims=True)
    SUB = np.hstack([Cm / np.maximum(tot, 1), np.log1p(Cm)]).astype(np.float32)   # 비율 + log 개수
    return dict(BIN=BIN, NMUT=np.asarray(BIN.sum(1)).ravel(), HOT=hot, WORDS=words, SUB=SUB)

TR, TE = featurize(V), featurize(Vt)
print('평균 변이 유전자 수 train', TR['NMUT'].mean().round(1))

평균 변이 유전자 수 train 35.3


## 4. 환자 중복 정리 (학습 데이터만)

- 변이가 완전히 같은 기록끼리 묶어서(`GRP`), 라벨이 두 개로 갈린 쌍이 20번 이상 나오면 "묶음 ↔ 세부" 관계로 봄 (KIPAN↔KIRC, GBMLGG↔LGG가 나옴)
- 그런 기록은 한 명으로 합치고 더 구체적인 **세부 라벨**을 붙임
- 변이가 하나도 없는 환자는 서로 다른 사람으로 둠
- 이 정리는 **학습 fold 안에서만** 함. test에는 아무것도 하지 않음

In [6]:
KEY = pd.util.hash_pandas_object(pd.DataFrame(V_raw), index=False).values.astype(np.int64)   # 변이 전체가 같으면 같은 값
GRP = KEY.copy(); z = np.flatnonzero(TR['NMUT'] == 0); GRP[z] = -(z + 1)                    # 변이 없는 환자는 각자 다른 묶음

def identity(d): return d                   # CountVectorizer에 이미 나눈 단어 목록을 그대로 넘기기 위함
def sub(L, ix): return [L[i] for i in ix]   # 리스트에서 ix 행만

def find_pairs(ix, min_pairs=20):
    labs = defaultdict(set)
    for i in ix: labs[GRP[i]].add(y[i])
    cnt = Counter(tuple(sorted(s)) for s in labs.values() if len(s) == 2); tot = Counter(y[ix]); pairs = {}
    for (a, b), c in cnt.items():
        if c >= min_pairs:
            spec, umb = (a, b) if tot[a] < tot[b] else (b, a)      # 환자가 적은 쪽이 세부 아형
            pairs[umb] = spec
    return pairs                                                   # {묶음: 세부}

def clean(ix, pairs):
    g = defaultdict(list)
    for i in ix: g[GRP[i]].append(i)
    keep, lab = [], []
    for rows in g.values():
        ls = {y[i] for i in rows}
        spec = [pairs[u] for u in ls if u in pairs and pairs[u] in ls] if len(ls) > 1 else []
        if spec: keep.append(rows[0]); lab.append(spec[0])         # 묶음/세부로 갈린 같은 변이 → 한 명, 세부 라벨
        else: keep += rows; lab += [y[i] for i in rows]
    return np.array(keep), np.array(lab)

print('묶음 → 세부:', {CL[u]: CL[s] for u, s in find_pairs(np.arange(N)).items()})

묶음 → 세부: {'KIPAN': 'KIRC', 'GBMLGG': 'LGG'}


## 5. 모델

- `fit`: 핫스팟·단어 사전을 학습 데이터에서만 만들고 XGBoost, TF-IDF 로지스틱 회귀, 2단계 로지스틱 회귀를 학습함 (모두 `balanced` 가중치: 환자가 적은 암도 똑같이 중요하게)
- `raw`: XGBoost와 로지스틱 회귀의 26개 아형 확률
- `proba`: 비중 학습 모델(`meta`)이 두 확률을 아형별로 섞은 최종 1단계 확률과, 2단계 "세부 아형일 확률"
- `fit_model`: 학습 데이터를 다시 3-fold로 나눠, 각 조각을 보지 않은 모델의 확률(52개)로 `meta`를 학습함 → 자기 정답을 본 확률로 배우는 누수를 막음

In [7]:
class Model:
    def fit(self, ix, lab, seed):
        self.hot = CountVectorizer(analyzer=identity, min_df=HOT_N, binary=True, lowercase=False).fit(sub(TR['HOT'], ix))
        self.cv = CountVectorizer(analyzer=identity, min_df=2, binary=True, lowercase=False).fit(sub(TR['WORDS'], ix))
        self.tf = TfidfTransformer(sublinear_tf=True).fit(self.cv.transform(sub(TR['WORDS'], ix)))
        sw = compute_sample_weight('balanced', lab)
        self.xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.3,
                                 tree_method='hist', n_jobs=-1, random_state=seed).fit(self._Xx(TR, ix), lab, sample_weight=sw)
        self.lr = LogisticRegression(C=LR_C, class_weight='balanced', max_iter=3000).fit(self._Xl(TR, ix), lab)
        self.gmask = np.flatnonzero(np.asarray(TR['BIN'][ix].sum(0)).ravel() >= 5)   # 2단계는 5명 이상 변이된 유전자만
        self.s2 = {}
        for umb, spec in self.pairs.items():                                         # 묶음 안에서 세부 아형인지
            m = np.isin(lab, [umb, spec])
            self.s2[(umb, spec)] = LogisticRegression(C=STAGE2_C, class_weight='balanced', max_iter=2000).fit(
                self._X2(TR, ix[m]), (lab[m] == spec).astype(int))
        return self
    def _Xx(self, S, ix):   # XGBoost 입력: 0/1 + 변이 수 + 핫스팟 + 치환쌍
        return sp.hstack([S['BIN'][ix], sp.csr_matrix(S['NMUT'][ix][:, None].astype(np.float32)),
                          self.hot.transform(sub(S['HOT'], ix)), sp.csr_matrix(S['SUB'][ix])], format='csr').astype(np.float32)
    def _Xl(self, S, ix):   # 텍스트 마이닝 입력: TF-IDF
        return self.tf.transform(self.cv.transform(sub(S['WORDS'], ix)))
    def _X2(self, S, ix):   # 2단계 입력: 자주 변이된 유전자 0/1 + log(변이 수)
        return sp.hstack([S['BIN'][ix][:, self.gmask], sp.csr_matrix(np.log1p(S['NMUT'][ix])[:, None])], format='csr')
    def raw(self, S, ix):
        def full(m, X):   # 모델이 모르는 아형 칸은 0으로 두고 26칸에 맞춤
            P = np.zeros((len(ix), len(CL))); P[:, m.classes_] = m.predict_proba(X); return P
        return np.hstack([full(self.xgb, self._Xx(S, ix)), full(self.lr, self._Xl(S, ix))])
    def proba(self, S, ix):
        P = np.zeros((len(ix), len(CL))); P[:, self.meta.classes_] = self.meta.predict_proba(self.raw(S, ix))
        Q = {k: m.predict_proba(self._X2(S, ix))[:, 1] for k, m in self.s2.items()}
        return P, Q

def fit_model(ix, seed):
    pairs = find_pairs(ix); k, l = clean(ix, pairs)
    M = Model(); M.pairs = pairs; M.fit(k, l, seed)
    Z = np.zeros((len(k), 2 * len(CL)))
    for itr, iva in StratifiedKFold(3, shuffle=True, random_state=seed).split(k, l):   # 학습 데이터 안 3-fold
        m_ = Model(); m_.pairs = pairs; m_.fit(k[itr], l[itr], seed); Z[iva] = m_.raw(TR, k[iva])
    M.meta = LogisticRegression(C=1.0, class_weight='balanced', max_iter=3000).fit(Z, l)   # 아형별 비중 학습
    return M

def decide(P, Q, pairs):
    # 2단계 판정: 묶음과 세부의 확률을 합쳐 먼저 고르고, 묶음이 뽑히면 세부일 확률 50% 기준으로 나눔
    Pm = P.copy()
    for umb, spec in pairs: Pm[:, umb] = P[:, umb] + P[:, spec]; Pm[:, spec] = -1
    pred = Pm.argmax(1)
    for (umb, spec), q in Q.items():
        m = pred == umb; pred[m] = np.where(q[m] >= 0.5, spec, umb)
    return pred

## 6. (선택) 5-fold 교차검증

- `RUN_CV=True`일 때만 실행함. fold마다 중복 정리·핫스팟·단어 사전·비중 학습을 모두 학습 fold 안에서 다시 함
- 참고 점수 (Macro F1, 시드 42): v12 5:5 섞기 0.5034 → 비중 학습 0.5068

In [8]:
if RUN_CV:
    oof = np.zeros(N, int); t0 = time.time()
    for f, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEEDS[0]).split(np.arange(N), y), 1):
        M = fit_model(tr, SEEDS[0]); P, Q = M.proba(TR, va); oof[va] = decide(P, Q, M.s2)
        print(f'fold {f}: {f1_score(y[va], oof[va], average="macro"):.4f} ({time.time() - t0:.0f}s)')
    print('OOF Macro F1:', round(f1_score(y, oof, average='macro'), 4))

## 7. 전체 train으로 학습 → test 예측 (시드 3개 평균)

- 시드마다 전체 train으로 `fit_model`을 학습하고 test 확률을 구함
- 1단계 확률과 2단계 확률을 각각 평균 낸 뒤 판정함
- 형제 라벨 규칙은 쓰지 않음

In [9]:
Ps, Qs, models = [], [], []; t0 = time.time(); te = np.arange(len(Vt))
for seed in SEEDS:
    M = fit_model(np.arange(N), seed); P, Q = M.proba(TE, te)
    Ps.append(P); Qs.append(Q); models.append(M)
    print(f'seed {seed} 학습 완료 ({time.time() - t0:.0f}s)')
P = np.mean(Ps, 0); Q = {k: np.mean([q[k] for q in Qs], 0) for k in Qs[0]}
pred = decide(P, Q, models[0].s2)
print('예측 아형 수:', len(set(pred)))

seed 42 학습 완료 (160s)


seed 7 학습 완료 (326s)


seed 123 학습 완료 (492s)
예측 아형 수: 26


## 8. 제출 파일 저장

- `sample_submission.csv`의 ID 순서 그대로 `SUBCLASS`를 채움
- 행 수·ID 순서·빈 값을 확인함

In [10]:
out = submission.copy(); out['SUBCLASS'] = le.inverse_transform(pred)
path = f'{OUT_DIR}/submission_v12_stackS3.csv'
out.to_csv(path, index=False, encoding='utf-8-sig')
chk = pd.read_csv(path)
assert len(chk) == len(test) and (chk['ID'] == test['ID']).all() and chk['SUBCLASS'].notna().all()
print('saved', path); print(chk['SUBCLASS'].value_counts().head(10))

saved output/submission_v12_stackS3.csv
SUBCLASS
STES     565
COAD     307
KIPAN    255
BRCA     109
OV       103
KIRC     102
LGG       95
PCPG      78
THYM      78
LIHC      71
Name: count, dtype: int64


## 9. 추론용 모델 파일 저장

- 데모(`04_demo.ipynb`)와 재현용으로 시드 3벌의 학습된 부품(XGBoost, 로지스틱 회귀, 단어 사전, idf, 비중 학습, 2단계)과 표기 통일 기준값을 한 파일로 저장함
- 파일: `output/model_v12_stackS3.pkl` (약 70MB). 다운로드 링크는 제출 메일/DM에 따로 첨부함
- 불러와서 예측하는 코드는 `scripts/oncova_v12_infer.py` (이 노트북과 같은 계산, test 2,500명 예측이 제출 파일과 100% 일치함을 확인함)

In [11]:
import joblib
def part(M):
    return dict(hot_vocab=M.hot.vocabulary_, word_vocab=M.cv.vocabulary_, idf=M.tf.idf_, xgb=M.xgb, lr=M.lr,
                meta=M.meta, s2=M.s2, gmask=M.gmask)
art = dict(gene_cols=genes, classes=CL, hot_cnt=dict(HOT_CNT), models=[part(M) for M in models], seeds=','.join(map(str, SEEDS)))
joblib.dump(art, f'{OUT_DIR}/model_v12_stackS3.pkl'); print('saved', f'{OUT_DIR}/model_v12_stackS3.pkl')

saved output/model_v12_stackS3.pkl
